# Etapa 3 — Probabilidade

Variáveis binárias (cada município é um "mundo"):
- **F** = tem facção em 2024 (`ALVO`)
- **V** = taxa de homicídio alta: terço mais alto da taxa por 100 mil em 2021–23 (`TXH = 2`)
- **E** = teve morte a esclarecer em 2021–23 (`ESC`)

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


In [2]:
F = d["ALVO"]; V = (d["TXH"] == 2).astype(int); E = d["ESC"]

## 1. Distribuição conjunta de 2 variáveis (4 eventos atômicos)

In [3]:
conj = pd.crosstab(V, F, normalize=True)
conj.index = ["V=0", "V=1"]; conj.columns = ["F=0", "F=1"]
print(conj.round(4)); print("Soma =", round(conj.values.sum(), 6))

        F=0     F=1
V=0  0.4876  0.1795
V=1  0.1918  0.1411
Soma = 1.0


## 2. Marginais e condicionais

In [4]:
pV, pF = V.mean(), F.mean()
pF_V1 = conj.loc["V=1", "F=1"] / conj.loc["V=1"].sum()
pF_V0 = conj.loc["V=0", "F=1"] / conj.loc["V=0"].sum()
print(f"P(V=1) = {pV:.4f}   P(F=1) = {pF:.4f}")
print(f"P(F=1 | V=1) = {pF_V1:.4f}   P(F=1 | V=0) = {pF_V0:.4f}   razão = {pF_V1 / pF_V0:.2f}x")

P(V=1) = 0.3329   P(F=1) = 0.3205
P(F=1 | V=1) = 0.4238   P(F=1 | V=0) = 0.2690   razão = 1.58x


## 3. Teorema de Bayes: raciocínio causal × diagnóstico

- **Causal:** P(V | F): a presença de facção **causa** violência. Dado que há facção, qual a chance de violência alta?
- **Diagnóstico:** P(F | V): observamos a violência (efeito) e inferimos a facção (causa oculta). É o que um analista de inteligência faz.

$$P(F \mid V) = \frac{P(V \mid F)\,P(F)}{P(V)}$$

In [5]:
pV_F1 = ((V == 1) & (F == 1)).sum() / (F == 1).sum()
bayes = pV_F1 * pF / pV
print(f"P(V=1 | F=1) [causal]      = {pV_F1:.4f}")
print(f"P(F=1 | V=1) via Bayes     = {pV_F1:.4f} x {pF:.4f} / {pV:.4f} = {bayes:.4f}")
print(f"P(F=1 | V=1) direto        = {pF_V1:.4f}  (confere)")

P(V=1 | F=1) [causal]      = 0.4402
P(F=1 | V=1) via Bayes     = 0.4402 x 0.3205 / 0.3329 = 0.4238
P(F=1 | V=1) direto        = 0.4238  (confere)


## 4. Regra da cadeia com 3 variáveis: P(F, V, E) = P(F) · P(V | F) · P(E | F, V)

In [6]:
linhas = []
for f in (0, 1):
    for v in (0, 1):
        for e in (0, 1):
            direto = ((F == f) & (V == v) & (E == e)).mean()
            cadeia = (F == f).mean() * (((V == v) & (F == f)).sum() / (F == f).sum()) \
                     * (((E == e) & (F == f) & (V == v)).sum() / ((F == f) & (V == v)).sum())
            linhas.append([f, v, e, direto, cadeia])
tab = pd.DataFrame(linhas, columns=["F", "V", "E", "P direta", "P pela cadeia"]).round(5)
print(tab.to_string(index=False)); print("Soma:", round(tab["P direta"].sum(), 6))

 F  V  E  P direta  P pela cadeia
 0  0  0   0.22401        0.22401
 0  0  1   0.26361        0.26361
 0  1  0   0.05322        0.05322
 0  1  1   0.13861        0.13861
 1  0  0   0.04827        0.04827
 1  0  1   0.13119        0.13119
 1  1  0   0.01733        0.01733
 1  1  1   0.12376        0.12376
Soma: 1.0


## 5. Independência condicional de 2 atributos dada a classe

$$P(V, E \mid F) \stackrel{?}{=} P(V \mid F)\,P(E \mid F)$$

In [7]:
from scipy.stats import chi2_contingency
for f in (0, 1):
    sub = F == f
    pVE = ((V[sub] == 1) & (E[sub] == 1)).mean()
    prod = V[sub].mean() * E[sub].mean()
    chi2, pval, *_ = chi2_contingency(pd.crosstab(V[sub], E[sub]))
    print(f"F={f}: P(V=1,E=1|F)={pVE:.4f}  P(V=1|F)P(E=1|F)={prod:.4f}  qui²={chi2:.1f}  p-valor={pval:.2e}")

F=0: P(V=1,E=1|F)=0.2040  P(V=1|F)P(E=1|F)=0.1671  qui²=14.5  p-valor=1.40e-04
F=1: P(V=1,E=1|F)=0.3861  P(V=1|F)P(E=1|F)=0.3501  qui²=7.5  p-valor=6.16e-03


**Leitura.** Medida como **taxa** (por 100 mil), a violência alta é um sinal mais fraco de facção (1,6 vez) do que era a contagem bruta, que também refletia o tamanho do município. Taxa alta e mortes a esclarecer **não** são independentes dada a facção. A hipótese do Naive Bayes é violada (Etapa 4).

## 6. Por que a lógica de primeira ordem não expressa "70% de chance"

A fórmula ∀x (ViolênciaAlta(x) → Facção(x)) é falsa, porque há municípios violentos sem facção registrada. A negação também é falsa. A LPO não tem como dizer "em parte dos casos": só a probabilidade expressa que P(F | V) é cerca de 0,42, contra 0,27 quando a taxa não é alta.

## 7. Transições entre 2024 e 2025 (4ª edição do Cartografias)

A 4ª edição permite olhar a **mudança**: quais municípios entraram, saíram ou passaram de domínio para disputa. É uma distribuição conjunta de duas variáveis com 3 valores cada (9 eventos atômicos).

In [8]:
rot = {0: "sem registro", 1: "uma facção", 2: "disputa"}
trans = pd.crosstab(d["REGIME"].map(rot), d["REGIME_2025"].map(rot), margins=True, margins_name="total")
print("Linhas = 2024, colunas = 2025"); print(trans)
print("\nP(regime 2025 | regime 2024):")
print(pd.crosstab(d["REGIME"].map(rot), d["REGIME_2025"].map(rot), normalize="index").round(3))

Linhas = 2024, colunas = 2025
REGIME_2025   disputa  sem registro  uma facção  total
REGIME                                                
disputa            54            13          18     85
sem registro       18           420         111    549
uma facção         16            32         126    174
total              88           465         255    808

P(regime 2025 | regime 2024):
REGIME_2025   disputa  sem registro  uma facção
REGIME                                         
disputa         0.635         0.153       0.212
sem registro    0.033         0.765       0.202
uma facção      0.092         0.184       0.724


In [9]:
sem = d[d["ALVO"] == 0]
pE = sem["ALVO_2025"].mean()
print(f"P(entrada em 2025 | sem facção em 2024) = {pE:.3f}")
for p in (0, 1, 2):
    s_ = sem[sem["PORTE"] == p]
    print(f"  P(entrada | PORTE={p}) = {s_['ALVO_2025'].mean():.3f}   (n={len(s_)})")
pP2_E = (sem.loc[sem["ALVO_2025"] == 1, "PORTE"] == 2).mean()
pP2 = (sem["PORTE"] == 2).mean()
print(f"\nBayes: P(PORTE=2 | entrada) = {pP2_E:.3f}  ->  P(entrada | PORTE=2) = {pP2_E:.3f} x {pE:.3f} / {pP2:.3f} = {pP2_E * pE / pP2:.3f}")

P(entrada em 2025 | sem facção em 2024) = 0.235
  P(entrada | PORTE=0) = 0.138   (n=240)
  P(entrada | PORTE=1) = 0.307   (n=290)
  P(entrada | PORTE=2) = 0.368   (n=19)

Bayes: P(PORTE=2 | entrada) = 0.054  ->  P(entrada | PORTE=2) = 0.054 x 0.235 / 0.035 = 0.368


**Leitura.** Entre os municípios sem registro em 2024, cerca de 1 em cada 4 passou a ter facção em 2025, e a chance cresce com o porte. Também há "saídas" (municípios que tinham facção em 2024 e não aparecem em 2025). O próprio Cartografias avisa que parte da mudança entre edições pode vir da **melhoria do método** e de mais operações policiais, e não só da expansão real.